In [ ]:
import torch
import torch.nn as nn
import torchvision.transforms as transforms
import torchvision.datasets as dsets
from torch import Tensor
import torch.nn.functional as F
from torch.nn import init, Parameter
import pdb
import math
import numpy as np
import torch.nn.functional as F
from torch.autograd import Variable
import os 
import random
import pandas as pd
from copy import deepcopy
from typing import Dict
import transformers
from torch import Tensor
from torch.nn import init, Parameter
import torch.nn.functional as F
import pdb
import math
from collections import defaultdict
import numpy as np
import os 
import json
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, recall_score, f1_score, average_precision_score, precision_score
from torch.utils.data import DataLoader,Dataset,TensorDataset
import torch.optim as optim

In [ ]:
hours_data = 24
data_path = "physionet.org/files/challenge-2012/1.0.0/phase1/all_subjects_data_48_hours/"
subject= np.load(os.path.join(data_path,"subjects_data.npz"),allow_pickle=True)
S = np.squeeze(subject["statics_data"], axis=1)
targets = subject["targets_data"]
timeseries = np.array(subject["timeseries_imp_data"], dtype=float)[:, :hours_data, :]
timeseries_org = np.array(subject["timeseries_data"], dtype=float)[:, :hours_data, :]
timedelta = np.array(subject["delta_time_data"], dtype=float)[:, :hours_data, :]
icu_mor = targets

In [ ]:
unique, counts = np.unique(icu_mor, return_counts=True)

dict(zip(unique, counts))

In [ ]:
repeats =timeseries.shape[1]
temporal_statics = np.tile(S, repeats).reshape(S.shape[0], repeats,S.shape[-1])
timedelta_statics= np.zeros((S.shape[0], repeats,S.shape[-1]))
temporal_statics.shape, timedelta_statics.shape

In [ ]:
temporal_data = np.concatenate((timeseries_org, temporal_statics), axis=-1)
temporal_timedelta = np.concatenate((timedelta, np.zeros_like(temporal_statics)), axis=-1)
temporal_timedelta[np.isnan(temporal_timedelta)] = 999
temporal_timedelta[np.isinf(temporal_timedelta)] = 999
temporal_data_features = np.concatenate((timeseries_org, temporal_statics), axis=-1)

In [ ]:
temporal_data_features

In [ ]:
timeseries.shape,timeseries_org.shape, temporal_data_features.shape

In [ ]:
temporal_data

In [ ]:
freq_list , timeseries_last_obs_data= [], []
nb_pats, seq, n_features = temporal_data.shape
for i in range(nb_pats):
    data_patient=  np.expand_dims(temporal_data[i,:,:], axis=0)
    nan_counts = np.sum(np.isnan(data_patient), axis=(0, 1))
    freq_list.append(np.repeat(np.expand_dims(nan_counts, axis=0), repeats, axis=0))
    # Last observation record
    Index_Last=(~np.isnan(temporal_data[i,:,:])).cumsum(0).argmax(0)
    Last_Indices = np.reshape(Index_Last,(1,n_features))
    Last_Values = np.take_along_axis(temporal_data[i,:,:], Last_Indices, axis = 0)
    timeseries_last_obs_data.append(np.repeat(Last_Values, repeats, axis=0))
freqs = np.stack(freq_list)
last_obs_data=np.stack(timeseries_last_obs_data)

In [ ]:
import numpy as np
import torch
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import KFold, train_test_split
from sklearn.model_selection import StratifiedKFold, StratifiedShuffleSplit
# ---------------------------------------------------------------------------
# Manual Min-Max Helpers (Global Temporal) with Clipping
# ---------------------------------------------------------------------------

def get_minmax_params_3d(x: np.ndarray):
    """Computes min and max for each feature across all N and T."""
    x_min = np.nanmin(x, axis=(0, 1), keepdims=True)
    x_max = np.nanmax(x, axis=(0, 1), keepdims=True)
    
    # Avoid division by zero
    x_max[x_max == x_min] += 1e-8
    return x_min, x_max

def apply_minmax_3d(x: np.ndarray, x_min: np.ndarray, x_max: np.ndarray) -> np.ndarray:
    """Applies scaling and clips values to the [0, 1] range."""
    normed = (x - x_min) / (x_max - x_min)
    return np.clip(normed, 0, 1) # Clipping added here

def get_minmax_params_2d(x: np.ndarray):
    """Computes min/max for 2D arrays (N, D)."""
    x_min = np.nanmin(x, axis=0, keepdims=True)
    x_max = np.nanmax(x, axis=0, keepdims=True)
    x_max[x_max == x_min] += 1e-8
    return x_min, x_max

def apply_minmax_2d(x: np.ndarray, x_min: np.ndarray, x_max: np.ndarray) -> np.ndarray:
    """Applies scaling and clips values to the [0, 1] range."""
    normed = (x - x_min) / (x_max - x_min)
    return np.clip(normed, 0, 1) # Clipping added here

def get_loaders(x, t, last, freq, y, batch_size, shuffle=False):
    """Wraps NumPy arrays into PyTorch DataLoaders."""
    dataset = TensorDataset(
        torch.tensor(x, dtype=torch.float32),
        torch.tensor(t, dtype=torch.float32),
        torch.tensor(last, dtype=torch.float32),
        torch.tensor(freq, dtype=torch.float32),
        torch.tensor(y, dtype=torch.float32),
    )
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle, num_workers=4)

# ---------------------------------------------------------------------------
# Cross-validation splitter
# ---------------------------------------------------------------------------

def cv_fold_splits_ehr(data, time_data, last_data, features_freqs, target, n_fold=10, batch_size=64):
    kfold = StratifiedKFold(n_splits=n_fold, shuffle=True, random_state=n_fold)
    test_data_loader, training_data, validation_data, fold_scalers = [], [], [], []

    for index, (train_index, test_index) in enumerate(kfold.split(data, target)):
        print(f'<------- OUTER FOLD {index + 1} (Manual MinMax + Clip) ------->')

        # ── Outer split ──
        x_train,      x_test      = data[train_index],           data[test_index]
        x_train_last, x_test_last = last_data[train_index],      last_data[test_index]
        x_train_freq, x_test_freq = features_freqs[train_index], features_freqs[test_index]
        x_train_t,    x_test_t    = time_data[train_index],      time_data[test_index]
        y_train,      y_test      = target[train_index],         target[test_index]

        # ── Inner split ──
        sss = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=n_fold+1)
        tr_idx, val_idx = next(sss.split(x_train, y_train))
        
        
        x_tr_in,    x_val_in    = x_train[tr_idx],       x_train[val_idx]
        t_tr_in,    t_val_in    = x_train_t[tr_idx],     x_train_t[val_idx]
        last_tr_in, last_val_in = x_train_last[tr_idx],  x_train_last[val_idx]
        freq_tr_in, freq_val_in = x_train_freq[tr_idx],  x_train_freq[val_idx]
        y_tr_in,    y_val_in    = y_train[tr_idx],       y_train[val_idx]

        # ── Fit: Calculate parameters strictly on inner-train ──
        x_min, x_max       = get_minmax_params_3d(x_tr_in)
        last_min, last_max = get_minmax_params_3d(last_tr_in) # Ensure last_data is 3D, else use 2D version

        # ── Transform: Apply Scaling + Clipping ──
        x_tr_in   = apply_minmax_3d(x_tr_in, x_min, x_max)
        x_val_in  = apply_minmax_3d(x_val_in, x_min, x_max)
        x_te_norm = apply_minmax_3d(x_test, x_min, x_max)

        last_tr_in  = apply_minmax_3d(last_tr_in, last_min, last_max)
        last_val_in = apply_minmax_3d(last_val_in, last_min, last_max)
        x_te_last   = apply_minmax_3d(x_test_last, last_min, last_max)

        fold_scalers.append({
            'x_params': {"x_min": x_min,
                        "x_max": x_max},
            'last_params': {"last_min": last_min,
                        "last_max": last_max}
        })

        # --- Step 5: Create DataLoaders ---
        train_loader = get_loaders(x_tr_in, t_tr_in, last_tr_in, freq_tr_in, y_tr_in, batch_size, shuffle=True)
        val_loader   = get_loaders(x_val_in, t_val_in, last_val_in, freq_val_in, y_val_in, batch_size, shuffle=False)
        test_loader  = get_loaders(x_te_norm, x_test_t, x_te_last,   x_test_freq, y_test, batch_size, shuffle=False)

        training_data.append(train_loader)
        validation_data.append(val_loader)
        test_data_loader.append(test_loader)

    return (test_data_loader, training_data, validation_data, fold_scalers)

In [ ]:
test_Dataloader, train_Dataloader, valid_Dataloader, fold_scalers = cv_fold_splits_ehr(temporal_data_features,
                                                                                       temporal_timedelta,
                                                                                       last_obs_data, 
                                                                                       freqs,icu_mor,
                                                                                       n_fold=10)

In [ ]:
dn=f"MIMICIII"
if not os.path.exists(dn):
    os.makedirs(dn)

seq_length = temporal_data_features.shape[1]
input_dim = temporal_data_features.shape[-1]
hidden_dim, output_dim  = 64, icu_mor.shape[-1]
taskname=f"PHYSIONET2012_INHOS_{seq_length}_HOURS_DATA_10_FOLDS".upper()
task=f"{os.path.join(dn, f'{taskname}')}"
if not os.path.exists(task):
    os.makedirs(task)
train_val_loader = random.sample(list(zip(train_Dataloader, valid_Dataloader, fold_scalers)),
                                 len(train_Dataloader))
np.savez(os.path.join(task, f"train_test_data.npz"), 
            folds_data_test= test_Dataloader,
            folds_data_train_valid= train_val_loader,)
        
np.savez(os.path.join(task, f"data_max_min.npz"), 
         seq_length=seq_length, input_dim=input_dim,
         output_dim=output_dim,data_stats=fold_scalers)
input_dim, seq_length, output_dim
task,seq_length, input_dim, hidden_dim, output_dim